# Data Download
Put the SampleSubmission.csv, Test.csv, and Train.csv under the same directory
I put it in /Data - Lucky


In [2]:
#Import some libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pylab import rcParams
import seaborn as sb
sb.set_style('darkgrid')
rcParams['figure.figsize'] = 8,8
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [3]:
#import data
train = pd.read_csv('Data/Train.csv')
test=  pd.read_csv('Data/Test.csv')
submission = pd.read_csv('Data/SampleSubmission.csv')

In [4]:
train.columns, test.columns

(Index(['user_id', 'REGION', 'TENURE', 'MONTANT', 'FREQUENCE_RECH', 'REVENUE',
        'ARPU_SEGMENT', 'FREQUENCE', 'DATA_VOLUME', 'ON_NET', 'ORANGE', 'TIGO',
        'ZONE1', 'ZONE2', 'MRG', 'REGULARITY', 'TOP_PACK', 'FREQ_TOP_PACK',
        'CHURN'],
       dtype='str'),
 Index(['user_id', 'REGION', 'TENURE', 'MONTANT', 'FREQUENCE_RECH', 'REVENUE',
        'ARPU_SEGMENT', 'FREQUENCE', 'DATA_VOLUME', 'ON_NET', 'ORANGE', 'TIGO',
        'ZONE1', 'ZONE2', 'MRG', 'REGULARITY', 'TOP_PACK', 'FREQ_TOP_PACK'],
       dtype='str'))

# The Problem
(Taken from https://zindi.world/competitions/expresso-churn-prediction)


Expresso is an African telecommunications company that provides customers with airtime and mobile data bundles. The objective of this challenge is to develop a machine learning model to predict the likelihood of each Expresso customer “churning,” i.e. becoming inactive and not making any transactions for 90 days.

This solution will help Expresso to better serve their customers by understanding which customers are at risk of leaving.

Variable Definition Tables
| Variable | Description | Data type |
|---|---|---|
| REGION | Client location | object |
| TENURE | Duration in the network | object |
| MONTANT | Top-up amount | float64 |
| FREQUENCE_RECH | Number of times the customer refilled | float64 |
| REVENUE | Monthly revenue generated by the client | float64 |
| ARPU_SEGMENT | Revenue over 90 days / 3 | float64 |
| FREQUENCE | Number of times the client generated revenue | float64 |
| DATA_VOLUME | Number of connections | float64 |
| ON_NET | Calls within Expresso | float64 |
| ORANGE | Calls to Orange | float64 |
| TIGO | Calls to Tigo | float64 |
| ZONE1 | Calls to zone 1 | float64 |
| ZONE2 | Calls to zone 2 | float64 |
| MRG | Unclear definition; constant column | object |
| REGULARITY | Number of times the client is active over 90 days | int64 |
| TOP_PACK | Client’s most frequently activated pack | object |
| FREQ_TOP_PACK | Number of times the client activated the top pack | float64 |
| CHURN | Variable to predict (target) | int64 |

Details:
1. Income*: refer to how much revenue did the customer generate for the company
2. MRG^: Unclearly defined variable, after checking, it is just a constant column.

In [5]:
# Checking Variable Definition
definitions = pd.read_csv('VariableDefinitions.csv', skiprows=1)
definitions = definitions.rename(columns={'Unnamed: 0': 'var name'})
definitions = definitions.drop(columns=["French"])
definitions.dropna(inplace=True)
definitions['dtype'] = definitions["var name"].map(
    train.dtypes.astype(str)

)
definitions

,var name,English,dtype
2,REGION,the location of each client,str
3,TENURE,duration in the network,str
4,MONTANT,top-up amount,float64
5,FREQUENCE_RECH,number of times the customer refilled,float64
6,REVENUE,monthly income of each client,float64
7,ARPU_SEGMENT,income over 90 days / 3,float64
8,FREQUENCE,number of times the client has made an income,float64
9,DATA_VOLUME,number of connections,float64
10,ON_NET,inter expresso call,float64
11,ORANGE,call to orange,float64


In [6]:
# Clarifying some of the unclear definitions: MRG and TOP_PACK

# What is: MRG?
for col in train.columns:
    num_unique = train[col].nunique()
    if num_unique <= 1:
        print(f"Constant column: {col}")

# What about TOP_PACK? Packet that the users bought the most
# We are probably going to turn this into multiple features
train['TOP_PACK'].head(20)

Constant column: MRG


0                On net 200F=Unlimited _call24H
1                                           NaN
2                       On-net 1000F=10MilF;10d
3                             Data:1000F=5GB,7d
4                   Mixt 250F=Unlimited_call24H
5     MIXT:500F= 2500F on net _2500F off net;2d
6                                           NaN
7                         All-net 500F=2000F;5d
8                                           NaN
9                            On-net 500F_FNF;3d
10                                          NaN
11               On net 200F=Unlimited _call24H
12                                          NaN
13                                          NaN
14                        All-net 500F=2000F;5d
15                         Data: 100 F=40MB,24H
16                                          NaN
17                                          NaN
18                         Data: 100 F=40MB,24H
19                            Data:1000F=5GB,7d
Name: TOP_PACK, dtype: str

# Data Split

In [7]:
target = "CHURN"
X, y = train.copy(), train[target].copy()

# Drop constant columns
# dropping user_id as well for obvious reasons
# NOT Dropping target YET, just to simplify deduplication and other stuff down the line
X.drop(columns=['user_id', 'MRG'], inplace=True)


## Data Understanding

### Handling Missing Values

In [8]:
# Count missing values per column
missing_counts = X.isna().sum()
has_na_cols = missing_counts > 0

missing_counts[has_na_cols]

REGION             849299
MONTANT            756739
FREQUENCE_RECH     756739
REVENUE            726048
ARPU_SEGMENT       726048
FREQUENCE          726048
DATA_VOLUME       1060433
ON_NET             786675
ORANGE             895248
TIGO              1290016
ZONE1             1984327
ZONE2             2017224
TOP_PACK           902594
FREQ_TOP_PACK      902594
dtype: int64

In [9]:
cols_with_na = [
    'REGION',
    'MONTANT',
    'FREQUENCE_RECH',
    'REVENUE',
    'ARPU_SEGMENT',
    'FREQUENCE',
    'DATA_VOLUME',
    'ON_NET',
    'ORANGE',
    'TIGO',
    'ZONE1',
    'ZONE2',
    'TOP_PACK',
    'FREQ_TOP_PACK'
]

# Helper function
def summarize_against_label(df, col, label):
    missing = df[col].isna()
    summary = (
        pd.DataFrame({
            f'{col}_is_missing': missing,
            'CHURN': label
        }).groupby(f"{col}_is_missing")['CHURN']
        .agg(customer="size", churn_rate="mean")
    )
    return summary

# Check one by one, kinda manual, but, eh, unescapable
# Purpose: Find whether "NaN" has a meaning or actually missing
for col in cols_with_na:
    summary = summarize_against_label(X, col, y)
    print(summary)
    print("=" * 50)

# Manually check: Essentially, EVERY NA columns has some association with churn rate
# Need to validate later, but, keeping every cols for now.
na_cols_with_importance = [
    'REGION',
    'MONTANT',
    'FREQUENCE_RECH',
    'REVENUE',
    'ARPU_SEGMENT',
    'FREQUENCE',
    'DATA_VOLUME',
    'ON_NET',
    'ORANGE',
    'TIGO',
    'ZONE1', # Very many missing values, has a 0.2 correlation rate with CHURN
    'ZONE2', # Same with ZONE1
    'TOP_PACK',
    'FREQ_TOP_PACK'
]

                   customer  churn_rate
REGION_is_missing                      
False               1304749    0.018020
True                 849299    0.447987
                    customer  churn_rate
MONTANT_is_missing                      
False                1397309    0.050035
True                  756739    0.441463
                           customer  churn_rate
FREQUENCE_RECH_is_missing                      
False                       1397309    0.050035
True                         756739    0.441463
                    customer  churn_rate
REVENUE_is_missing                      
False                1428000    0.054057
True                  726048    0.450098
                         customer  churn_rate
ARPU_SEGMENT_is_missing                      
False                     1428000    0.054057
True                       726048    0.450098
                      customer  churn_rate
FREQUENCE_is_missing                      
False                  1428000    0.054057
True   

In [10]:
# is 0 used for numeric columns?
num_cols = X.select_dtypes(include=np.number).columns
for col in num_cols:
    print(f"Column {col}: {X[col].describe()['min']}")

Column MONTANT: 10.0
Column FREQUENCE_RECH: 1.0
Column REVENUE: 1.0
Column ARPU_SEGMENT: 0.0
Column FREQUENCE: 1.0
Column DATA_VOLUME: 0.0
Column ON_NET: 0.0
Column ORANGE: 0.0
Column TIGO: 0.0
Column ZONE1: 0.0
Column ZONE2: 0.0
Column REGULARITY: 1.0
Column FREQ_TOP_PACK: 1.0
Column CHURN: 0.0


In [11]:
# Deal with Missing Values
# Since each columns has some level of association:
# Numeric column -> 0
# We can use 0 here because it does indicate the user has no activity in that area, which is the assumption
# Categorical column - None
# No need for ColumnTransformer, as these are constant replacement of NaNs

# I know this includes the target column, but, no Na values in the target column, so, no worries
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(exclude=np.number).columns

num_cols, cat_cols
X[num_cols] = X[num_cols].fillna(0)
X[cat_cols] = X[cat_cols].fillna("None")

In [12]:
# No More NaN values!
missing_counts = X.isna().sum()
has_na_cols = missing_counts > 0

missing_counts[has_na_cols]

Series([], dtype: int64)

### Any Duplicates?


In [13]:
print(f"Duplicated rows count = {X.duplicated().sum()}")
# Note -> YES, A lot of duplications. But, this are different customers...
# How does each "duplication group" compare with the churn label?

# Exact duplicates include the label; feature duplicates exclude it.
total_duplicates = X.duplicated().sum()
feature_cols = X.columns.drop(target).tolist()
duplicate_mask = X.duplicated(subset=feature_cols, keep=False)
duplicate_rows = X.loc[duplicate_mask].copy()

# Count both labels for each repeated feature combination.
duplicate_summary = (
    duplicate_rows.groupby(feature_cols, dropna=False, observed=True, sort=False)[target]
    .agg(rows="size", distinct_labels="nunique", churn_1="sum")
)
duplicate_summary["churn_0"] = duplicate_summary["rows"] - duplicate_summary["churn_1"]
conflicting_groups = duplicate_summary.loc[
    duplicate_summary["distinct_labels"] > 1
].sort_values("rows", ascending=False)

print(f"Rows in repeated feature groups (including first occurrences): {len(duplicate_rows):,}")
print(f"Repeated feature groups: {len(duplicate_summary):,}")
print(f"Groups with consistent labels: {(duplicate_summary['distinct_labels'] == 1).sum():,}")
print(f"Groups with conflicting labels: {len(conflicting_groups):,}")
print(f"Rows in conflicting groups: {conflicting_groups['rows'].sum():,}")

# Each displayed group has identical features but both CHURN=0 and CHURN=1.
display(conflicting_groups.head(10).reset_index())


Duplicated rows count = 661897
Rows in repeated feature groups (including first occurrences): 692,284
Repeated feature groups: 21,753
Groups with consistent labels: 13,119
Groups with conflicting labels: 8,634
Rows in conflicting groups: 633,275


,REGION,TENURE,MONTANT,FREQUENCE_RECH,REVENUE,ARPU_SEGMENT,FREQUENCE,DATA_VOLUME,ON_NET,ORANGE,TIGO,ZONE1,ZONE2,REGULARITY,TOP_PACK,FREQ_TOP_PACK,rows,distinct_labels,churn_1,churn_0
0,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,None,0.0,133708,2,105059,28649
1,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,None,0.0,72200,2,49960,22240
2,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,None,0.0,45608,2,28196,17412
3,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,None,0.0,31621,2,18087,13534
4,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,None,0.0,23153,2,12080,11073
5,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6,None,0.0,17679,2,8544,9135
6,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7,None,0.0,13980,2,6256,7724
7,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8,None,0.0,10968,2,4691,6277
8,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9,None,0.0,8761,2,3538,5223
9,None,K > 24 month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10,None,0.0,7448,2,2827,4621


# Working Data

In [14]:
# These duplicates are irreducible noise
# What do we do with irreducible noise? Keep them all.
# CARE: Need to do hashing to ensure that isolation between train and valid sets.
train_y = X[target]
train_X = X.drop(columns=[target]).copy()
hashes = pd.util.hash_pandas_object(train_X, index=False)
len(train_X), len(hashes), len(hashes.unique())

(2154048, 2154048, 1483517)

# Exploratory Data Analysis
Let's just check around the data a bit

In [15]:
# Clearer view on the irreducible noise
g = y.groupby(hashes)

print(g.nunique().value_counts()) # For each hash, how many unique labels are thehre
print(g.var().mean()) # Get mean variance. Rows with 1 unique label has 0 variance.

CHURN
1    1474883
2       8634
Name: count, dtype: int64
0.12889918151635607


In [16]:
# Explanation: Around 0.5% of the rows have conflicting labels.
# Since this is a binary classification problem, the mean variance here indicates that:
# p * (1-p) = 0.1289 --> irreducible noise among duplicate features.
8634 / (1474883 + 8634)

0.005819953529349512

# Side Tangent: Trying to parse TOP_PACK

In [17]:
train_packs = set(train["TOP_PACK"].dropna())
test_packs = set(test["TOP_PACK"].dropna())

only_in_train = train_packs - test_packs
only_in_test = test_packs - train_packs

# Looks doable to parse. Let's do it?
len(only_in_train), len(only_in_test)

(40, 2)

In [18]:
train_packs

{'1000=Unlimited7Day',
 '1500=Unlimited7Day',
 '150=unlimited pilot auto',
 '200=Unlimited1Day',
 '200=unlimited pilot auto',
 '200F=10mnOnNetValid1H',
 '301765007',
 '305155009',
 '500=Unlimited3Day',
 'APANews_monthly',
 'APANews_weekly',
 'All-net 1000=5000;5d',
 'All-net 1000F=(3000F On+3000F Off);5d',
 'All-net 300=600;2d',
 'All-net 5000= 20000off+20000on;30d',
 'All-net 500= 4000off+4000on;24H',
 'All-net 500F =2000F_AllNet_Unlimited',
 'All-net 500F=1250F_AllNet_1250_Onnet;48h',
 'All-net 500F=2000F;5d',
 'All-net 500F=4000F ; 5d',
 'All-net 600F= 3000F ;5d',
 'CVM_100F_unlimited',
 'CVM_100f=200 MB',
 'CVM_100f=500 onNet',
 'CVM_150F_unlimited',
 'CVM_200f=400MB',
 'CVM_500f=2GB',
 'CVM_On-net 1300f=12500',
 'CVM_On-net 400f=2200F',
 'CVM_on-net bundle 500=5000',
 'Data: 100 F=40MB,24H',
 'Data: 200 F=100MB,24H',
 'Data: 200F=1GB,24H',
 'Data: 490F=Night,00H-08H',
 'Data:1000F=2GB,30d',
 'Data:1000F=5GB,7d',
 'Data:1000F=700MB,7d',
 'Data:1500F=3GB,30D',
 'Data:1500F=SPPackage

In [19]:
PACK_TYPES = [
    'data',
    'on-net',
    'all-net',
    'mixt',
    'internat',
    'wifi',
    'jokko'
]
VALIDITY = [
    ('day', 'daily'),
    ('week', 'weekly'),
    ('month', 'monthly'),
]
UNLIMITED_FLAG = [
    'unlimited', 'illimite' # illimiate = french for unlimited
]

def pack_type_parser(pack_name):
    pack_name = str(pack_name).lower()
    for pack_type in PACK_TYPES:
        if pack_type in pack_name:
            return pack_type
    return "other" # None is used for no value, using other to state that "user bought something"

def validity_parser(pack_name):
    pack_name = str(pack_name).lower()
    for validity in VALIDITY:
        if validity[0] in pack_name:
            return validity[1] # return full validity name
    return "other"

def unlimited_flag_parser(pack_name):
    pack_name = str(pack_name).lower()
    for flag in UNLIMITED_FLAG:
        if flag in pack_name:
            return True
    return False

In [21]:
# To try: without parsing vs drop raw + parsing vs leave raw + parsing
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import TargetEncoder
from sklearn.tree import DecisionTreeClassifier

temp_X, temp_y = train_X.copy(), train_y.copy()

# Parsed features: pack_type, validity, unlimited_flag
for func in [pack_type_parser, validity_parser, unlimited_flag_parser]:
    col_name = "_".join(func.__name__.split("_")[:-1])
    temp_X[col_name] = temp_X["TOP_PACK"].apply(func)

parsed_cols = ["pack_type", "validity", "unlimited_flag"]

# Every string column that needs encoding, across all three runs
all_cats = list(dict.fromkeys(list(cat_cols) + ["TOP_PACK", "pack_type", "validity"]))

# One grouping for all three runs, hashed without TOP_PACK so the folds are
# identical and safe for the parsed-only run too.
temp_hashes = pd.util.hash_pandas_object(temp_X.drop(columns="TOP_PACK"), index=False)

cv = GroupKFold(n_splits=5)


def score(X):
    target_cols = [c for c in all_cats if c in X.columns]
    pipe = make_pipeline(
        make_column_transformer(
            (TargetEncoder(), target_cols), # Just for simplicity sake, We could probably play around with this to maximize performance.
            remainder="passthrough",
        ),
        DecisionTreeClassifier(max_depth=8, random_state=42),
    )
    return cross_val_score(
        pipe, X, temp_y, cv=cv, groups=temp_hashes, scoring="roc_auc", n_jobs=5
    )


cv_scores_raw = score(temp_X.drop(columns=parsed_cols))
cv_scores_raw_parse = score(temp_X)
cv_scores_parse_only = score(temp_X.drop(columns="TOP_PACK"))
cv_scores_no_top_pack = score(temp_X.drop(columns=["TOP_PACK", "pack_type", "validity", "unlimited_flag"]))
print(f"""\
Raw scores: {cv_scores_raw.mean():.3f} (+/- {cv_scores_raw.std() * 2:.3f})
Raw + parsed scores: {cv_scores_raw_parse.mean():.3f} (+/- {cv_scores_raw_parse.std() * 2:.3f})
Parsed only scores: {cv_scores_parse_only.mean():.3f} (+/- {cv_scores_parse_only.std() * 2:.3f})
No TOP_PACK scores: {cv_scores_no_top_pack.mean():.3f} (+/- {cv_scores_no_top_pack.std() * 2:.3f})
""")

print("Raw+parsed minus raw, per fold: ", cv_scores_raw_parse - cv_scores_raw)
print("Parsed-only minus raw, per fold:", cv_scores_parse_only - cv_scores_raw)

Raw scores: 0.920 (+/- 0.024)
Raw + parsed scores: 0.920 (+/- 0.025)
Parsed only scores: 0.920 (+/- 0.024)
No TOP_PACK scores: 0.920 (+/- 0.024)

Raw+parsed minus raw, per fold:  [ 1.65056826e-05 -8.10908696e-05 -8.12131888e-04 -8.63151950e-05
  1.40012337e-04]
Parsed-only minus raw, per fold: [ 8.30576760e-07 -1.21859198e-04 -2.42351274e-05 -5.15348912e-05
  1.81725682e-04]
